# Update Commodity Demand in `nodes_*.json`

Reads per-region annual demand CSVs and writes `AggregatedDemandConstraint` values
into the matching node instances in `nodes_1.json` through `nodes_7.json`.

**Commodities covered:** Cement, CrudeSteel, Aluminum, Ammonia, Methanol

**Stage → year mapping** is loaded from `../stage_year_map.csv`.

In [26]:
import csv
import json
from pathlib import Path

import pandas as pd

In [27]:
# ── Step 1: Config & paths ────────────────────────────────────────────────────
# node type → (demand CSV filename, node ID prefix used in instance IDs)
COMMODITY_CONFIG = {
    "Aluminum":   ("aluminum_demand_5yeartimesteps.csv",  "aluminum"),
    "CrudeSteel": ("steel_demand_5yeartimesteps.csv",     "crudesteel"),
    "Cement":     ("cement_demand_5yeartimesteps.csv",    "cement"),
    "Ammonia":    ("ammonia_demand_5yeartimesteps.csv",   "ammonia"),
    "Methanol":   ("methanol_demand_5yeartimesteps.csv",  "methanol"),
}

demand_dir = Path(".").resolve()  # this notebook's directory (annual_commmodity_demand/)
nodes_dir  = demand_dir.parent    # system/

print("Demand CSV directory:", demand_dir)
print("Nodes JSON directory:", nodes_dir)


Demand CSV directory: /Users/al3792/Documents_Local/NZC_June_2026/NZC_Workshop_Example_Systems_June_2026/NZC_Workshop_All Sector_12day_MultiPeriods_WIP/case/system/annual_commmodity_demand
Nodes JSON directory: /Users/al3792/Documents_Local/NZC_June_2026/NZC_Workshop_Example_Systems_June_2026/NZC_Workshop_All Sector_12day_MultiPeriods_WIP/case/system


In [28]:
# ── Step 2: Load stage → year mapping ────────────────────────────────────────
stage_year_map = (
    pd.read_csv(nodes_dir / "stage_year_map.csv")
    .set_index("stage_index")["year"]
    .to_dict()
)

print("Stage → Year mapping:")
for stage, year in stage_year_map.items():
    print(f"  nodes_{stage}.json  →  {year}")


Stage → Year mapping:
  nodes_1.json  →  2030
  nodes_2.json  →  2035
  nodes_3.json  →  2040
  nodes_4.json  →  2045
  nodes_5.json  →  2050
  nodes_6.json  →  2055
  nodes_7.json  →  2060


In [30]:
# ── Step 3: Load demand CSVs ──────────────────────────────────────────────────
# Each CSV has regions as rows and years (2025–2060) as columns.
# We drop 2025 implicitly — only years in stage_year_map are ever looked up.

demand_data = {}
for node_type, (csv_file, _) in COMMODITY_CONFIG.items():
    df = pd.read_csv(demand_dir / csv_file, index_col=0)
    df.columns = [int(c) for c in df.columns]   # cast "2030" → 2030
    demand_data[node_type] = df
    print(f"  {node_type:<12}  {len(df)} regions  |  years: {list(df.columns)}")


  Aluminum      31 regions  |  years: [2025, 2030, 2035, 2040, 2045, 2050, 2055, 2060]
  CrudeSteel    31 regions  |  years: [2025, 2030, 2035, 2040, 2045, 2050, 2055, 2060]
  Cement        31 regions  |  years: [2025, 2030, 2035, 2040, 2045, 2050, 2055, 2060]
  Ammonia       31 regions  |  years: [2025, 2030, 2035, 2040, 2045, 2050, 2055, 2060]
  Methanol      31 regions  |  years: [2025, 2030, 2035, 2040, 2045, 2050, 2055, 2060]


In [38]:
# ── Step 4: Update & write ────────────────────────────────────────────────────
# Only instance_data[*].rhs_policy.AggregatedDemandConstraint is touched.
# All other keys (global_data, other rhs_policy entries, etc.) are left as-is.

# Show one instance before writing so you can verify nothing else changes
with open(nodes_dir / "nodes_1.json") as f:
    _peek = json.load(f)
for node in _peek["nodes"]:
    if node["type"] == "Aluminum":
        print("BEFORE — nodes_1.json, first Aluminum instance:")
        print(json.dumps(node["instance_data"][0], indent=2))
        break

print()

for stage_idx, year in stage_year_map.items():
    nodes_path = nodes_dir / f"nodes_{stage_idx}.json"
    with open(nodes_path) as f:
        data = json.load(f)

    updated = 0
    for node in data.get("nodes", []):
        node_type = node.get("type")
        if node_type not in demand_data:
            continue
        df = demand_data[node_type]
        for instance in node.get("instance_data", []):
            region = instance["id"].split("_", 1)[1]  # "aluminum_Region1Beijing" → "Region1Beijing"
            if region not in df.index:
                print(f"WARNING: '{region}' not found in {node_type} CSV")
                continue
            # Only this one value is changed; every other key is untouched
            instance["rhs_policy"]["AggregatedDemandConstraint"] = float(df.loc[region, year])
            updated += 1

    with open(nodes_path, "w") as f:
        json.dump(data, f, indent=2)
    print(f"nodes_{stage_idx}.json  (year {year}):  {updated} values written")

print()

# Confirm the same instance after — only the demand number should differ
with open(nodes_dir / "nodes_1.json") as f:
    _peek = json.load(f)
for node in _peek["nodes"]:
    if node["type"] == "Aluminum":
        print("AFTER — nodes_1.json, first Aluminum instance:")
        print(json.dumps(node["instance_data"][0], indent=2))
        break


BEFORE — nodes_1.json, first Aluminum instance:
{
  "id": "aluminum_Region1Beijing",
  "rhs_policy": {
    "AggregatedDemandConstraint": 58617.0
  }
}

nodes_1.json  (year 2030):  155 values written
nodes_2.json  (year 2035):  155 values written
nodes_3.json  (year 2040):  155 values written
nodes_4.json  (year 2045):  155 values written
nodes_5.json  (year 2050):  155 values written
nodes_6.json  (year 2055):  155 values written
nodes_7.json  (year 2060):  155 values written

AFTER — nodes_1.json, first Aluminum instance:
{
  "id": "aluminum_Region1Beijing",
  "rhs_policy": {
    "AggregatedDemandConstraint": 58617.0
  }
}
